MCP client integration

In [ ]:
!pip install -U langchain-mcp-adapters mcp==1.24.0

In [ ]:
!pip install -U langchain langchain-google-genai

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage
from google.colab import userdata
from langchain.agents import create_agent

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

In [ ]:
comp_api=userdata.get("composio_api")
kaggle_api=userdata.get("kaggle_api")

client = MultiServerMCPClient(
    {
        "Indian company fundamentals": {
            "transport": "http",
            "url": "https://www.kaggle.com/mcp",
            "headers": {
                "Authorization": f"Bearer {kaggle_api}"
            },
        }
    }
)

creating agent

In [ ]:
gemini_api=userdata.get("GEMINI_API_KEY")

In [ ]:
model=init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=gemini_api
)

In [ ]:
async def access_tools_mcp():
  all_tools= await client.get_tools()
  agent = create_agent(
      model=model,
      tools=all_tools,
      system_prompt="you are an assistant who gives answer to finance related questions for a company by the user",
      debug=True,
  )
  user_query = "What is the share value of apple"

  response = await agent.ainvoke({
    "messages": [{"role": "user", "content": user_query}]
  })
  print(response["messages"][-1].content[0]["text"])
await access_tools_mcp()

In [ ]:
!pip install stoxim-api

In [ ]:
%%capture x
!pip install -qU --no-cache-dir unsloth unsloth_zoo pyarrow datasets
!pip install -qU --no-deps trl peft accelerate bitsandbytes xformers

In [ ]:
from unsloth import FastModel
from transformers import TextStreamer
from google.colab import files
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

In [ ]:

model,tokenizer=FastModel.from_pretrained(
    model_name="unsloth/Qwen3-4b",
    max_seq_length=2048,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)

In [ ]:
def helper_func(messages,new_tokens=256):
  _=model.generate(
   **tokenizer.apply_chat_template(
       messages,
       add_generation_prompt=True,
       tokenize=True,
       return_dict=True,
       enable_thinking=False,
       return_tensors="pt",
   ).to("cuda"),
   max_new_tokens=new_tokens,
   temperature=1.0,
   top_p=0.97,
   top_k=65,
   streamer=TextStreamer(tokenizer,skip_prompt=True)
  )

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers = False,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_gradient_checkpointing = "unsloth",
)

In [ ]:

uploaded = files.upload()

In [ ]:

dataset=load_dataset(
    "json",
    data_files="first_chapter_examples_intinv_clean_strict.jsonl",
    split="train"
)

In [ ]:
format_texts = []

for i in range(len(dataset)):

    if dataset[i]["input"].strip():
        user_prompt = (
            f"Instruction:\n{dataset[i]["instruction"]}\n\n"
            f"Input:\n{dataset[i]["input"]}"
        )
    else:
        user_prompt = dataset[i]["instruction"]

    conversation = [
        {
            "role": "user",
            "content": user_prompt,
        },
        {
            "role": "assistant",
            "content": dataset[i]["output"],
        },
    ]

    text = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=False,
    )

    format_texts.append(text)

# Check if the 'text' column already exists and remove it before adding
if "text" in dataset.column_names:
    dataset = dataset.remove_columns("text")

dataset = dataset.add_column("text", format_texts)

In [ ]:


trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    max_seq_length=2048,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 5,
        packing=True,
        learning_rate = 5e-5,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
trainer.train()

In [ ]:
model=FastModel.for_inference(model)
helper_func([{"role": "user", "content":"are you finetuned on the book intelligent investor"}])

In [ ]:
print(len(dataset))
print(dataset[0])